In [ ]:
import numpy as np
import pandas as pd

In [ ]:
fold0 = pd.read_csv(f"/scratch1/smaruj/suppressing_CTCFs/results/fold0_with_positions_steps_results.tsv", sep="\t")

fold0["fold"] = [0 for i in range(len(fold0))]

In [ ]:
fold1 = pd.read_csv(f"/scratch1/smaruj/suppressing_CTCFs/results/fold1_with_positions_steps_results.tsv", sep="\t")

fold1["fold"] = [1 for i in range(len(fold1))]

In [ ]:
fold2 = pd.read_csv(f"/scratch1/smaruj/suppressing_CTCFs/results/fold2_with_positions_steps_results.tsv", sep="\t")

fold2["fold"] = [2 for i in range(len(fold2))]

In [ ]:
fold3 = pd.read_csv(f"/scratch1/smaruj/suppressing_CTCFs/results/fold3_with_positions_steps_results.tsv", sep="\t")

fold3["fold"] = [3 for i in range(len(fold3))]

In [ ]:
fold4 = pd.read_csv(f"/scratch1/smaruj/suppressing_CTCFs/results/fold4_with_positions_steps_results.tsv", sep="\t")

fold4["fold"] = [4 for i in range(len(fold4))]

In [ ]:
fold5 = pd.read_csv(f"/scratch1/smaruj/suppressing_CTCFs/results/fold5_with_positions_steps_results.tsv", sep="\t")

fold5["fold"] = [5 for i in range(len(fold5))]

In [ ]:
fold6 = pd.read_csv(f"/scratch1/smaruj/suppressing_CTCFs/results/fold6_with_positions_steps_results.tsv", sep="\t")

fold6["fold"] = [6 for i in range(len(fold6))]

In [ ]:
fold7 = pd.read_csv(f"/scratch1/smaruj/suppressing_CTCFs/results/fold7_with_positions_steps_results.tsv", sep="\t")

fold7["fold"] = [7 for i in range(len(fold7))]

In [ ]:
df = pd.concat([fold0, fold1, fold2,
                fold3, fold4, fold5,
                fold6, fold7], ignore_index=True)

In [ ]:
df["URQ_diff"] = df["URQ_result"] - df["URQ_init"]

In [ ]:
# selecting only sequences with a measurable contact enrichment
df = df[df['URQ_diff'] > 0.0]

In [ ]:
df.columns

In [ ]:
import ast

# Convert stringified sets into real Python sets of tuples
df["orig_CTCFs_coord"] = df["orig_CTCFs_coord"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df["new_CTCFs_coord"] = df["new_CTCFs_coord"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

In [ ]:
rows = []

for _, row in df.iterrows():
    chrom = row["chrom"]
    fold = row["fold"]
    centered_start = row["centered_start"]
    centered_end = row["centered_end"]
    
    # coords = row["orig_CTCFs_coord"]
    coords = row["new_CTCFs_coord"]
    
    if not coords:  # skip empty sets
        continue
    
    for coord in coords:
        if isinstance(coord, tuple) and len(coord) == 3:
            start, end, orientation = coord
            rows.append({
                "chrom": chrom,
                "fold": fold,
                "centered_start": centered_start,
                "centered_end": centered_end,
                "ctcf_start": start,
                "ctcf_end": end,
                "orientation": orientation
            })

ctcf_df = pd.DataFrame(rows)

In [ ]:
ctcf_df

In [ ]:
ctcf_df[ctcf_df["ctcf_end"] > 2048]

In [ ]:
# ctcf_df.to_csv("/scratch1/smaruj/suppressing_CTCFs/results/preexisting_ctcf_df.tsv", sep="\t", index=False)
ctcf_df.to_csv("/scratch1/smaruj/suppressing_CTCFs/results/new_ctcf_df.tsv", sep="\t", index=False)

# pred

In [1]:
import torch
import numpy as np
import pandas as pd
from Bio import SeqIO

In [2]:
import sys
import os
sys.path.append(os.path.abspath("/home1/smaruj/pytorch_akita/"))

from model_v2_compatible import SeqNN

In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [4]:
model = SeqNN()
model.load_state_dict(torch.load("/home1/smaruj/pytorch_akita/model_0_v2_finetuned_correctly.pt", map_location=device))
model.eval()

/tmp/SLURM_1764243/ipykernel_3609571/2744654124.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("/home1/smaruj/pytorch_akita/model_0_v2_

SeqNN(
  (stochastic_reverse_complement): StochasticReverseComplement()
  (stochastic_shift): StochasticShift()
  (conv_block_1): ConvBlock(
    (conv): Conv1d(4, 128, kernel_size=(15,), stride=(1,), padding=(7,), bias=False)
    (batch_norm): BatchNorm1d(128, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_tower): ConvTower(
    (conv_tower): Sequential(
      (0): ReLU()
      (1): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): ReLU()
      (5): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (6): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (7): MaxPool1d(kernel_size=2, stride=2, paddi

In [5]:
def one_hot_encode(seq, alphabet="ACGT"):
    """One-hot encode a DNA sequence into (len(seq), 4) numpy array."""
    mapping = {base: i for i, base in enumerate(alphabet)}
    arr = np.zeros((len(seq), len(alphabet)), dtype=np.float32)
    for i, base in enumerate(seq.upper()):
        if base in mapping:
            arr[i, mapping[base]] = 1.0
    return arr

In [6]:
# Load ctcf table
df = pd.read_csv(
    "/scratch1/smaruj/suppressing_CTCFs/results/preexisting_ctcf_df.tsv", 
    sep="\t"
)

In [7]:
# Load background fasta (first sequence only)
background_fasta = "/scratch1/smaruj/background_generation/background_sequences_scd30_totvar1300.fasta"
background_record = next(SeqIO.parse(background_fasta, "fasta"))
background_seq = str(background_record.seq)
print(f"Background length: {len(background_seq)}")

Background length: 1310720


In [8]:
# One-hot encode background
background_1hot = one_hot_encode(background_seq)
background_tensor = torch.tensor(background_1hot.T).unsqueeze(0).to(device)  # shape (1, 4, L)

In [9]:
# Baseline prediction
with torch.no_grad():
    baseline_pred = model(background_tensor).cpu()

In [ ]:
df

In [10]:
df[df["ctcf_start"] < 0]

,chrom,fold,centered_start,centered_end,ctcf_start,ctcf_end,orientation
192,chr7,1,116893696,118204416,-45,-26,+
236,chr14,2,34529280,35840000,-18,1,-
284,chr3,2,64485376,65796096,-42,-23,-
335,chr11,3,52287488,53598208,-55,-36,-
346,chr12,3,15458304,16769024,-7,12,-
350,chr12,3,15458304,16769024,-17,2,-
408,chr15,3,40192000,41502720,-4,15,-
410,chr15,3,60985344,62296064,-9,10,+
576,chr10,5,83523584,84834304,-3,16,-
653,chr12,6,39186432,40497152,-15,4,+


In [11]:
df[df["ctcf_end"] > 2048]

,chrom,fold,centered_start,centered_end,ctcf_start,ctcf_end,orientation
41,chr4,0,127655936,128966656,2075,2094,+
65,chr5,0,97955840,99266560,2031,2050,-
157,chr13,1,39806976,41117696,2057,2076,+
750,chr11,7,92659712,93970432,2031,2050,-
797,chr19,7,51900416,53211136,2080,2099,+


In [12]:
def reverse_complement(ohe_seq):
    """
    Reverse complement of one-hot encoded sequence.
    Input shape: (B, 4, L) or (4, L).
    Returns same shape.
    """
    if ohe_seq.ndim == 2:  # (4, L) → add batch dim
        ohe_seq = ohe_seq.unsqueeze(0)

    # Reverse along sequence axis (last dimension)
    rc = torch.flip(ohe_seq, dims=[-1])

    # Complement: swap A<->T, C<->G along channel axis (dim=1)
    rc = rc[:, [3, 2, 1, 0], :]

    if rc.shape[0] == 1:  # if we added batch dim, squeeze it back
        rc = rc.squeeze(0)

    return rc

In [13]:

import os
from tqdm import tqdm

extra_flank = 60
bin_size = 2048
center_bin = 320  # central bin index

slice_start = bin_size * center_bin
slice_end = bin_size * (center_bin+1)

collected_ctcfs = []

for fold in range(1):
    fold_df = df[df["fold"] == fold]
    
    for _, row in tqdm(fold_df.iterrows(), total=len(fold_df), desc=f"Fold {fold}"):
        chrom = row["chrom"]
        cstart = row["centered_start"]
        cend = row["centered_end"]
        ctcf_start = row["ctcf_start"]
        ctcf_end = row["ctcf_end"]
        orientation = row["orientation"]

        # Path to OHE file
        path = f"/scratch1/smaruj/suppressing_CTCFs/results/fold{fold}/{chrom}_{cstart}_{cend}_slice.pt"
        if not os.path.exists(path):
            print(f"⚠️ Missing: {path}")
            continue

        # Load OHE tensor (4, L)
        slice_ohe = torch.load(path).to(device)
        
        if ctcf_start >= 0 and ctcf_end <= 2048:
            # fully inside slice → take directly
            ctcf_seq = slice_ohe[:, :, ctcf_start:ctcf_end].squeeze(0)
            
        else:
            parts = []

            full_path = f"/scratch1/smaruj/suppressing_CTCFs/ohe_X/fold{fold}/{chrom}_{cstart}_{cend}_X.pt"
            if not os.path.exists(full_path):
                print(f"⚠️ Missing: {full_path}")
                continue

            # Load OHE tensor (4, L)
            full_ohe = torch.load(full_path).to(device)
            
            # left overhang
            if ctcf_start < 0:
                # how much overhang
                left_len = -ctcf_start
                # take from full sequence before slice_start
                parts.append(full_ohe[:, :, slice_start+ctcf_start : slice_start])

                # take the inside part from slice (from 0 to ctcf_end)
                parts.append(slice_ohe[:, :, 0:ctcf_end])

            # right overhang
            elif ctcf_end > 2048:
                right_len = ctcf_end - 2048
                # take the inside part from slice (from ctcf_start to 2048)
                parts.append(slice_ohe[:, :, ctcf_start:2048])
                # take from full sequence after slice_end
                parts.append(full_ohe[:, :, slice_end : slice_end+right_len])

            ctcf_seq = torch.cat(parts, dim=2)

        # Reverse complement if needed
        if orientation == "-":
            ctcf_seq = reverse_complement(ctcf_seq)
        
        collected_ctcfs.append(ctcf_seq)

Fold 0:   0%|          | 0/89 [00:00<?, ?it/s]/tmp/SLURM_1764243/ipykernel_3609571/1885752452.py:31: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  slice_ohe = torch.load(pat

In [14]:
batch_size = 16
scd_scores = []

batch_tensors = []
batch_indices = []

for idx, ctcf_ohe in enumerate(collected_ctcfs):  
    # Ensure shape is (4, L_ctcf) (remove batch dim if present)
    if ctcf_ohe.ndim == 3:  
        ctcf_ohe = ctcf_ohe[0]

    # Clone background (shape (4, L_bg))
    inserted_tensor = background_tensor.clone()

    # Insert position (middle of background)
    insert_pos = inserted_tensor.shape[2] // 2
    frag_len = ctcf_ohe.shape[1]

    start = insert_pos - frag_len // 2
    end = start + frag_len
    
    # Insert CTCF sequence
    inserted_tensor[:, :, start:end] = ctcf_ohe.unsqueeze(0)

    # Collect for batching
    batch_tensors.append(inserted_tensor.to(device))
    batch_indices.append(idx)

    # Process batch
    if len(batch_tensors) == batch_size or idx == len(collected_ctcfs) - 1:
        batch_input = torch.cat(batch_tensors, dim=0)  # (B, 4, L)
        with torch.no_grad():
            batch_pred = model(batch_input).cpu()

        # Compute SCD
        for i, pred in enumerate(batch_pred):
            diff = pred - baseline_pred.squeeze(0)
            scd = torch.sqrt(torch.sum(diff ** 2)).item()
            scd_scores.append((batch_indices[i], scd))

        # Reset
        batch_tensors, batch_indices = [], []


In [15]:
scd_scores

[(0, 34.83784484863281),
 (1, 36.538108825683594),
 (2, 6.684276103973389),
 (3, 48.25735092163086),
 (4, 0.06844861060380936),
 (5, 42.57271194458008),
 (6, 37.63166809082031),
 (7, 46.603206634521484),
 (8, 38.31870651245117),
 (9, 34.19162368774414),
 (10, 48.76954650878906),
 (11, 0.1700889766216278),
 (12, 36.59212112426758),
 (13, 24.083621978759766),
 (14, 39.870914459228516),
 (15, 19.4370059967041),
 (16, 46.39826965332031),
 (17, 27.524648666381836),
 (18, 45.79581069946289),
 (19, 41.88452911376953),
 (20, 46.21815490722656),
 (21, 40.44673156738281),
 (22, 53.773765563964844),
 (23, 48.830772399902344),
 (24, 45.2310676574707),
 (25, 0.06578181684017181),
 (26, 48.43452072143555),
 (27, 46.96391296386719),
 (28, 28.891948699951172),
 (29, 46.73124313354492),
 (30, 0.14526166021823883),
 (31, 47.87738800048828),
 (32, 0.19379769265651703),
 (33, 40.57752990722656),
 (34, 47.4362907409668),
 (35, 41.290428161621094),
 (36, 42.90800094604492),
 (37, 32.38676834106445),
 (38, 3